In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Configuración predeterminada y opciones de configuración de la transpilación


<details>
<summary><b>Versiones de paquetes</b></summary>

El código de esta página fue desarrollado con los siguientes requisitos.
Recomendamos usar estas versiones o versiones más recientes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Los circuitos abstractos deben transpilarse porque las QPUs tienen un conjunto limitado de puertas base y no pueden ejecutar operaciones arbitrarias. La función del transpilador es modificar circuitos arbitrarios para que puedan ejecutarse en una QPU específica. Esto se hace traduciendo los circuitos a las puertas base admitidas e introduciendo puertas SWAP según sea necesario, de modo que la conectividad del circuito coincida con la de la QPU.

Como se explica en [Transpilar con gestores de pasadas](transpile-with-pass-managers), puedes crear un [gestor de pasadas](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) usando la función [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) y pasar un circuito o una lista de circuitos a su método [run](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) para transpilarlos. Puedes llamar a `generate_preset_pass_manager` pasando únicamente el nivel de optimización y el backend, y usar los valores predeterminados para todas las demás opciones, o puedes pasar argumentos adicionales a la función para ajustar la transpilación.

## Uso básico sin parámetros
En este ejemplo, pasamos un circuito y la QPU de destino al transpilador sin especificar ningún parámetro adicional.

Crea un circuito y visualiza el resultado:

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Create circuit to test transpiler on
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

# Add measurements to the circuit
qc.measure_all()

# View the circuit
qc.draw(output="mpl")

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg)

Transpila el circuito y visualiza el resultado:

In [2]:
from qiskit.transpiler import generate_preset_pass_manager

# Specify the QPU to target
backend = FakeSherbrooke()

# Transpile the circuit
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
transpiled_circ = pass_manager.run(qc)

# View the transpiled circuit
transpiled_circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg)

## Todos los parámetros disponibles
A continuación se presentan todos los parámetros disponibles para la función [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager). Hay dos clases de argumentos: los que describen el objetivo de la compilación y los que influyen en el funcionamiento del transpilador.

Todos los parámetros excepto `optimization_level` son opcionales. Para más detalles, consulta la [documentación de la API del Transpiler](https://docs.quantum.ibm.com/api/qiskit/transpiler#transpiler-api).

- `optimization_level` (int) - Cuánta optimización se realiza sobre los circuitos. Entero en el rango (0 - 3). Los niveles más altos generan circuitos más optimizados, a costa de un mayor tiempo de transpilación. Consulta [Establecer el nivel de optimización del transpilador](set-optimization) para más detalles.

### Parámetros para describir el objetivo de compilación:
Estos argumentos describen la QPU de destino para la ejecución del circuito, incluida información como el mapa de acoplamiento de la QPU (que describe la conectividad de los qubits), las puertas base admitidas por la QPU y las tasas de error de las puertas.

Muchos de estos parámetros se describen en detalle en [Parámetros de uso común para la transpilación](common-parameters).

<details>
  <summary>
    **Parámetros de QPU (`Backend`)**
  </summary>

**Parámetros de Backend** - Si especificas `backend`, no necesitas especificar `target` ni ninguna otra opción de backend. Del mismo modo, si especificas `target`, no necesitas especificar `backend` ni ninguna otra opción de backend.
- `backend` (Backend) - Si se establece, el transpilador compila el circuito de entrada para este dispositivo. Si se establece cualquier otra opción que afecte a esta configuración, como `coupling_map`, esta anula la configuración de `backend`.
- `target` (Target) - Un objetivo de transpilación del backend. Normalmente se especifica como parte del argumento de backend, pero si construiste manualmente un objeto Target, puedes especificarlo aquí. Esto anula el objetivo de `backend`.
- `backend_properties` (BackendProperties) - Propiedades devueltas por una QPU, que incluyen información sobre errores de puertas, errores de lectura, tiempos de coherencia de qubits, entre otros. Encuentra una QPU que proporcione esta información ejecutando `backend.properties()`.
- `timing_constraints` (Dict[str, int] | None) - Una restricción opcional del hardware de control sobre la resolución temporal de las instrucciones. Esta información la proporciona la configuración de la QPU. Si la QPU no tiene ninguna restricción sobre la asignación de tiempo de instrucción, `timing_constraints` es `None` y no se realiza ningún ajuste. Una QPU puede reportar un conjunto de restricciones, a saber:
    - `granularity`: Un valor entero que representa la resolución mínima de la puerta de pulso en unidades de dt. Una puerta de pulso definida por el usuario debe tener una duración que sea múltiplo de este valor de granularidad.
    - `min_length`: Un valor entero que representa la longitud mínima de la puerta de pulso en unidades de dt. Una puerta de pulso definida por el usuario debe ser más larga que esta longitud.
    - `pulse_alignment`: Un valor entero que representa la resolución temporal del tiempo de inicio de una instrucción de puerta. Las instrucciones de puerta deben comenzar en un tiempo que sea múltiplo de este valor.
    - `acquire_alignment`: Un valor entero que representa la resolución temporal del tiempo de inicio de una instrucción de medición. La instrucción de medición debe comenzar en un tiempo que sea múltiplo de este valor.
</details>

<details>
  <summary>
    **Parámetros de layout y topología**
  </summary>

- `basis_gates` (List[str] | None) - Lista de nombres de puertas base a las que desenrollar. Por ejemplo ['u1', 'u2', 'u3', 'cx']. Si es `None`, no se desenrolla.
- `coupling_map` (CouplingMap | List[List[int]]) - Mapa de acoplamiento dirigido (posiblemente personalizado) al que apuntar en el mapeo. Si el mapa de acoplamiento es simétrico, es necesario especificar ambas direcciones. Se admiten los siguientes formatos:
    - Instancia de CouplingMap
    - Lista - debe proporcionarse como una matriz de adyacencia, donde cada entrada especifica todas las interacciones de dos qubits dirigidas admitidas por la QPU. Por ejemplo: [[0, 1], [0, 3], [1, 2], [1, 5], [2, 5], [4, 1], [5, 3]]
- `inst_map` (List[InstructionScheduleMap] | None) - Mapeo de operaciones de circuito a programas de pulsos. Si es `None`, se usa el `instruction_schedule_map` de la QPU.
</details>

### Parámetros para influir en el funcionamiento del transpilador
Estos parámetros afectan a etapas específicas de la transpilación. Algunos pueden afectar a múltiples etapas, pero se han listado bajo una sola etapa por simplicidad. Si especificas un argumento, como `initial_layout` para los qubits que deseas usar, ese valor anula todas las pasadas que podrían cambiarlo. En otras palabras, el transpilador no cambiará nada que especifiques manualmente. Para obtener detalles sobre etapas específicas, consulta [Etapas del transpilador](transpiler-stages).

<details>
  <summary>
    **Etapa de inicialización**
  </summary>

- `hls_config` (HLSConfig) - Una clase de configuración opcional `HLSConfig` que se pasa directamente a la pasada de transformación `HighLevelSynthesis`. Esta clase de configuración te permite especificar las listas de algoritmos de síntesis y sus parámetros para varios objetos de alto nivel.
- `init_method` (str) - El nombre del plugin a usar para la etapa de inicialización. Por defecto, no se usa un plugin externo. Puedes ver una lista de plugins instalados ejecutando `list_stage_plugins()` con `init` como argumento del nombre de etapa.
- `unitary_synthesis_method` (str) - El nombre del método de síntesis unitaria a usar. Por defecto, se usa `default`. Puedes ver una lista de plugins instalados ejecutando `unitary_synthesis_plugin_names()`.
- `unitary_synthesis_plugin_config` (dict) - Un diccionario de configuración opcional que se pasa directamente al plugin de síntesis unitaria. Por defecto, esta configuración no tiene efecto porque el método de síntesis unitaria predeterminado no acepta configuración personalizada. Aplicar una configuración personalizada solo es necesario cuando se especifica un plugin de síntesis unitaria con el argumento `unitary_synthesis`. Como esto es específico de cada plugin de síntesis unitaria, consulta la documentación del plugin para saber cómo usar esta opción.
</details>

<details>
  <summary>
    **Etapa de layout**
  </summary>

- `initial_layout` (Layout | Dict | List) - Posición inicial de los qubits virtuales sobre los qubits físicos. Si este layout hace que el circuito sea compatible con las restricciones del `coupling_map`, se usará. No se garantiza que el layout final sea el mismo, ya que el transpilador puede permutar qubits mediante SWAPs u otros medios. Para más detalles, consulta la [sección de layout inicial.](common-parameters#initial-layout)
- `layout_method` (str) - Nombre de la pasada de selección de layout (`default`, `dense`, `sabre` y `trivial`). También puede ser el nombre del plugin externo a usar para la etapa de layout. Puedes ver una lista de plugins instalados ejecutando `list_stage_plugins()` con `layout` como argumento `stage_name`. El valor predeterminado es `sabre`.
</details>

<details>
  <summary>
    **Etapa de enrutamiento**
  </summary>

- `routing_method` (str) - Nombre de la pasada de enrutamiento (`basic`, `lookahead`, `default`, `sabre` o `none`). También puede ser el nombre del plugin externo a usar para la etapa de enrutamiento. Puedes ver una lista de plugins instalados ejecutando `list_stage_plugins()` con `routing` como argumento `stage_name`. El valor predeterminado es `sabre`.
</details>

<details>
  <summary>
    **Etapa de traducción**
  </summary>

- `translation_method` (str) - Nombre de la pasada de traducción (`default`, `synthesis`, `translator`, `ibm_backend`, `ibm_dynamic_circuits`, `ibm_fractional`). También puede ser el nombre del plugin externo a usar para la etapa de traducción. Puedes ver una lista de plugins instalados ejecutando `list_stage_plugins()` con `translation` como argumento `stage_name`. El valor predeterminado es `translator`.
</details>

<details>
  <summary>
    **Etapa de optimización**
  </summary>

- `approximation_degree` (float, en el rango 0-1 | None) - Dial heurístico usado para la aproximación de circuitos (1.0 = sin aproximación, 0.0 = aproximación máxima). El valor predeterminado es 1.0. Especificar `None` establece el grado de aproximación a la tasa de error reportada. Consulta la [sección de grado de aproximación](common-parameters#approx-degree) para más detalles.
- `optimization_method` (str) - El nombre del plugin a usar para la etapa de optimización. Por defecto no se usa un plugin externo. Puedes ver una lista de plugins instalados ejecutando `list_stage_plugins()` con `optimization` como argumento `stage_name`.
</details>

<details>
  <summary>
    **Etapa de planificación**
  </summary>

- `scheduling_method` (str) - Nombre de la pasada de planificación. También puede ser el nombre del plugin externo a usar para la etapa de planificación. Puedes ver una lista de plugins instalados ejecutando `list_stage_plugins()` con `scheduling` como argumento `stage_name`.
  * 'as_soon_as_possible': Planifica las instrucciones de forma ávida, lo más pronto posible en un recurso de qubit (alias: `asap`).
  * 'as_late_as_possible': Planifica las instrucciones tarde, es decir, manteniendo los qubits en el estado fundamental cuando sea posible (alias: `alap`). Este es el valor predeterminado.
</details>

<details>
  <summary>
    **Ejecución del transpilador**
  </summary>

- `seed_transpiler` (int) - Establece semillas aleatorias para las partes estocásticas del transpilador.
</details>

Los siguientes valores predeterminados se usan si no especificas ninguno de los parámetros anteriores. Consulta la [página de referencia de la API](../api/qiskit/transpiler_preset) del método para más información:

In [3]:
generate_preset_pass_manager(
    optimization_level=1,
    backend=None,
    target=None,
    basis_gates=None,
    coupling_map=None,
    initial_layout=None,
    layout_method=None,
    routing_method=None,
    translation_method=None,
    scheduling_method=None,
    approximation_degree=1.0,
    seed_transpiler=None,
    unitary_synthesis_method="default",
    unitary_synthesis_plugin_config=None,
    hls_config=None,
    init_method=None,
    optimization_method=None,
)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [Set the optimization level](set-optimization).
    - Review more [Commonly used parameters](common-parameters).
    - Learn how to [Set the optimization level when using Qiskit Runtime.](./runtime-options-overview)
    - Visit the [Transpile with pass managers](transpile-with-pass-managers) topic.
    - For examples, see [Representing quantum computers.](./represent-quantum-computers)
    - Learn [how to transpile circuits](/docs/guides/circuit-transpilation-settings) as part of the Qiskit patterns workflow using Qiskit Runtime.
    - Review the [Transpile API documentation](/docs/api/qiskit/transpiler).
</Admonition>